# HW06 · Перенос обучения и ML-системы / Transfer learning and ML systems

100 баллов: код 80 + объяснения 20. / 100 points: code 80 + explanations 20.
Срок — живая матрица группы. После срока: raw × max(0,1−просрочка/(14 суток)). / Deadline: live group matrix. Afterwards: raw × max(0,1−lateness/(14 days)).

## Данные и протокол / Data and protocol
**CIFAR-10**: реальные цветные изображения 32×32, 10 классов. Объект — одно изображение; признаки — RGB-пиксели, доступные при запросе; цель — категория объекта. / Real 32×32 colour images in 10 classes. One observation is one image; RGB pixels are available at prediction time; the target is the object category.
Источник / Source: https://www.cs.toronto.edu/~kriz/cifar.html (Krizhevsky, 2009). Официальные части: 50 000 train, 10 000 test. / Official partitions: 50,000 train and 10,000 test.
Train и Validation берём без пересечений из официального train; Test — отдельная фиксированная подвыборка official test. Метки Test используются только для фиксированного стратифицированного отбора и финальной оценки. / Train and Validation are disjoint subsets of official train; Test is a fixed stratified subset of official test. Test labels are used only for predefined stratified sampling and final scoring.
Нет ID источника/автора или времени: нельзя утверждать перенос на новых авторов либо будущие данные. Маленькая учебная подвыборка даёт шумную оценку, не результат на полном benchmark. Возможное пересечение источников ImageNet и CIFAR здесь не проверено. / No source/writer/time IDs: this does not establish generalization to unseen authors or future data. Small-subset scores are noisy, not full-benchmark results. Source overlap between ImageNet and CIFAR has not been audited here.
Первый запуск скачивает CIFAR-10 (~163 MiB) и веса (~10 MiB). Интернет нужен для скачивания; платного API нет. CPU достаточно; извлечение признаков может занять несколько минут. / First run downloads data (~163 MiB) and weights (~10 MiB). Internet is needed for downloads; no paid API. CPU is sufficient; feature extraction may take several minutes.


In [ ]:
STUDENT_NAME = ""
STUDENT_GROUP = ""
READY = False  # True после выполнения T1–T5 / after completing T1–T5

In [ ]:
# В Colab библиотеки обычно установлены. Если import не проходит, выполните эту ячейку и перезапустите среду.
# Colab normally includes these packages. If imports fail, run this cell and restart the runtime.
# %pip install torch torchvision scikit-learn pandas matplotlib pillow


In [ ]:
import copy, json, time, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torchvision, sklearn
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import CIFAR10
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
torch.set_num_threads(2)
print({'python':platform.python_version(), 'torch':torch.__version__, 'torchvision':torchvision.__version__, 'numpy':np.__version__, 'sklearn':sklearn.__version__})


In [ ]:
official_train = CIFAR10('data', train=True, download=True)
official_test = CIFAR10('data', train=False, download=True)
classes = list(official_train.classes)
y_source = np.asarray(official_train.targets)
y_test_source = np.asarray(official_test.targets)
weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
backbone = mobilenet_v3_small(weights=weights)
backbone.classifier = nn.Identity()  # 576 признаков вместо 1000 ImageNet-классов / 576 features, not ImageNet labels
backbone.requires_grad_(False)
backbone.eval()
assert all(not p.requires_grad for p in backbone.parameters())
print(preprocess)


## T1 · Split · 15
Для каждого класса 0…9 последовательно перемешайте np.flatnonzero(train_labels==label), затем test_labels через один default_rng(seed). Первые n_train -> Train, следующие n_val -> Validation; первые n_test из test -> Test. Сохраняйте порядок классов и перемешанных индексов. / For each class 0…9, permute train IDs then test IDs using one default_rng(seed). Slice Train, Validation and Test exactly as stated; preserve class and shuffled order.

In [ ]:
def split_indices(train_labels, test_labels, n_train, n_val, n_test, seed):
    raise NotImplementedError('Complete this task')


## T2 · Признаки / Features · 20
Верните ndarray (len(ids),576), в порядке ids. model.eval(), torch.inference_mode(), DataLoader shuffle=False; никакого обучения. Используйте переданный transform. / Return features in ids order. Use eval, inference_mode, no shuffle, supplied transform; do not train the model.

In [ ]:
def extract_features(dataset, ids, model, transform, batch_size=32):
    raise NotImplementedError('Complete this task')


## T3 · Классификатор / Head · 15
StandardScaler + LogisticRegression(C=C,max_iter=2000,random_state=2026); fit только на переданных Train features/labels. / Fit the specified pipeline only on supplied training features and labels.

In [ ]:
def fit_head(features, labels, C):
    raise NotImplementedError('Complete this task')


## T4 · Выбор / Selection · 15
Из непустого records выберите C по максимуму val_macro_f1; точное равенство -> меньший C. Вход не менять. / Select C by max validation macro-F1; exact ties -> smaller C. Do not mutate input.

In [ ]:
def choose_run(records):
    raise NotImplementedError('Complete this task')


## T5 · Компонент приложения / Prediction component · 15
Принимайте только uint8 RGB (32,32,3), иначе ValueError. Одинаковый transform, eval и inference_mode. Верните dict: class_id:int, label:str, probabilities:dict с 10 именами классов и float. Используйте head.classes_ для соответствия столбцов именам. / Accept only uint8 RGB (32,32,3), otherwise ValueError. Use the same transform, eval and inference_mode; return class_id, label and a 10-class probability dictionary, mapping columns via head.classes_.

In [ ]:
def predict_image(image, model, transform, head, names):
    raise NotImplementedError('Complete this task')


In [ ]:
def pixels(dataset, ids):
    return dataset.data[ids].reshape(len(ids),-1).astype(np.float32)/255.0

def report(y, pred):
    return {'accuracy':float(accuracy_score(y,pred)), 'macro_f1':float(f1_score(y,pred,average='macro',zero_division=0))}


## Заданный эксперимент / Required experiment
100/30/30 на класс, seed=2027. Не меняйте C, данные, split и метрику после просмотра Test. / 100/30/30 per class, seed=2027. Do not change C, data, split or metric after viewing Test.

In [ ]:
if READY:
    SEED = 2027
    train_ids, val_ids, test_ids = split_indices(y_source, y_test_source, 100, 30, 30, SEED)
    assert len(set(train_ids)&set(val_ids)) == 0
    assert [len(x) for x in (train_ids,val_ids,test_ids)] == [1000,300,300]
    # Test IDs относятся к другому массиву, числовое совпадение индексов не означает пересечения.
    # Test IDs index a separate array: equal integer IDs do not imply overlapping observations.
    y_train, y_val, y_test = y_source[train_ids], y_source[val_ids], y_test_source[test_ids]
    assert all(set(v)==set(range(10)) for v in (y_train,y_val,y_test))
    fig, axes = plt.subplots(2,5,figsize=(10,4))
    for k, ax in enumerate(axes.flat):
        ax.imshow(official_train.data[train_ids[np.flatnonzero(y_train==k)[0]]]); ax.set_title(classes[k]); ax.axis('off')
    plt.tight_layout(); plt.show()
    
    baseline = fit_head(pixels(official_train,train_ids),y_train,0.1)
    print('Pixel baseline validation:', report(y_val,baseline.predict(pixels(official_train,val_ids))))
    before = {k:v.clone() for k,v in backbone.state_dict().items()}
    start = time.perf_counter()
    F_train = extract_features(official_train,train_ids,backbone,preprocess)
    F_val = extract_features(official_train,val_ids,backbone,preprocess)
    feature_seconds = time.perf_counter()-start
    assert F_train.shape == (1000,576) and F_val.shape == (300,576)
    assert np.isfinite(F_train).all() and np.isfinite(F_val).all()
    assert all(torch.equal(before[k],v) for k,v in backbone.state_dict().items())
    print('Feature extraction seconds:',feature_seconds)
    records, heads = [], {}
    for C in [0.01,0.1,1.0]:
        heads[C] = fit_head(F_train,y_train,C)
        records.append({'C':C, 'val_macro_f1':report(y_val,heads[C].predict(F_val))['macro_f1']})
    display(pd.DataFrame(records))
    selected_C = choose_run(records)
    head = heads[selected_C]
    selection = {'C':selected_C,'criterion':'validation macro-F1; exact tie -> smaller C',
                 'weights':'MobileNet_V3_Small_Weights.IMAGENET1K_V1','seed':SEED}
    print('FROZEN BEFORE TEST:', selection)
    
    # Test доступен только после фиксации выбора / Access Test only after freezing selection
    F_test = extract_features(official_test,test_ids,backbone,preprocess)
    test_results = {'pixels':report(y_test,baseline.predict(pixels(official_test,test_ids))),
                    'transfer':report(y_test,head.predict(F_test))}
    print('FINAL TEST:',test_results)
    ConfusionMatrixDisplay(confusion_matrix(y_test,head.predict(F_test)),display_labels=classes).plot(xticks_rotation=60)
    plt.show()
    
    image = official_train.data[val_ids[0]]
    response = predict_image(image,backbone,preprocess,head,classes)
    assert response['class_id'] == int(head.predict(F_val[:1])[0])
    assert np.allclose(list(response['probabilities'].values()),head.predict_proba(F_val[:1])[0],atol=1e-5)
    assert np.isclose(sum(response['probabilities'].values()),1)
    print('Service response:',response)
    # Прогрев, затем 10 полных вызовов на Validation / Warm-up, then 10 end-to-end validation calls
    predict_image(image,backbone,preprocess,head,classes)
    timings=[]
    for i in range(10):
        t=time.perf_counter(); predict_image(official_train.data[val_ids[i]],backbone,preprocess,head,classes)
        timings.append((time.perf_counter()-t)*1000)
    print('CPU end-to-end median ms:',float(np.median(timings)))
    experiment_log = {**selection,'versions':{'torch':torch.__version__,'torchvision':torchvision.__version__,'sklearn':sklearn.__version__},
                      'train_ids':train_ids.tolist(),'val_ids':val_ids.tolist(),'test_ids':test_ids.tolist(),
                      'class_names':classes,'preprocess':str(preprocess),'test_results':test_results,
                      'median_latency_ms':float(np.median(timings)), 'latency_scope':'warm CPU preprocess+backbone+head, batch=1'}
    Path('experiment_hw06.json').write_text(json.dumps(experiment_log,ensure_ascii=False,indent=2))
else:
    print("Complete tasks and set READY=True")

## Объяснения / Explanations · 20
1. **5**: источник, объект, доступность признаков, counts Train/Validation/Test и ограничение переноса. / Source, observation, feature availability, split counts and a generalization limitation.
2. **5**: таблица трёх C, обоснование выбора, Test baseline/transfer; преимущество не обязательно. / Three-C table, selection rationale and baseline/transfer test results; improvement is not required.
3. **5**: ошибка Validation (или отсутствие): изображение и ID, истинный и предсказанный класс, score; гипотеза следующего эксперимента без настройки по Test. / Validation error (or absence): image and ID, true/predicted class and score; a follow-up hypothesis without Test tuning.
4. **5**: полный latency и условия измерения; один риск расхождения training/serving; отличие freeze/fine-tuning; источники и AI-помощь. / Full latency and measurement conditions; one training/serving mismatch risk; freeze vs fine-tuning; sources and AI assistance.
**Ваш ответ / Your answer:** …


In [ ]:
# Самопроверка структурная, не итоговая оценка / Structural checks, not a grade
if READY:
    assert len(set(train_ids))==1000 and len(set(val_ids))==300 and len(set(test_ids))==300
    assert not (set(train_ids) & set(val_ids))
    assert np.array_equal(np.bincount(y_train,minlength=10),np.full(10,100))
    assert np.array_equal(np.bincount(y_val,minlength=10),np.full(10,30))
    assert np.array_equal(np.bincount(y_test,minlength=10),np.full(10,30))
    assert F_train.shape==(1000,576) and F_val.shape==(300,576)
    assert all(torch.equal(before[k],v) for k,v in backbone.state_dict().items())
    assert isinstance(head[0],StandardScaler) and isinstance(head[1],LogisticRegression)
    assert np.allclose(head[0].mean_,F_train.mean(axis=0),atol=1e-5)
    probe=[{'C':1.,'val_macro_f1':.8},{'C':.1,'val_macro_f1':.8}]
    original=copy.deepcopy(probe)
    assert choose_run(probe)==.1 and probe==original
    for bad in [np.zeros((32,32),dtype=np.uint8),np.zeros((32,32,3),dtype=float),np.zeros((16,16,3),dtype=np.uint8)]:
        try: predict_image(bad,backbone,preprocess,head,classes)
        except ValueError: pass
        else: raise AssertionError('Invalid input must raise ValueError')
    assert set(response['probabilities'])==set(classes)
    assert np.isclose(sum(response['probabilities'].values()),1)
    print('Structural checks passed; instructor reviews all requirements and explanations.')
